In [4]:
import random
from torchvision.datasets import OxfordIIITPet
import numpy as np
import torch
from pathlib import Path
import math
from matplotlib import pyplot as plt
from PIL import Image
from tqdm import tqdm
import pandas as pd

def get_device() -> torch.device:
    """Resolve the best available torch device, preferring CUDA, then Apple MPS, then CPU.

    Returns:
        torch.device: the selected device for model inference/training.
    """
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def set_global_seed(seed: int) -> None:
    """Seed Python, NumPy and PyTorch RNGs for reproducible runs.

    Args:
        seed: the seed value applied to all random number generators.
    """
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

DATA_ROOT = Path("data")
MANIFESTS_DIR = Path("manifests")
MANIFESTS_DIR.mkdir(exist_ok=True)
SEED = 42
DEVICE = get_device()

set_global_seed(SEED)

print(f"Using device: {DEVICE}")

Using device: mps


In [5]:
import json

SPLITS_DIR = Path("splits")
TRAIN_VAL_SPLIT_PATH = SPLITS_DIR / "train_val_indices.json"
SELECTED_VARIANTS_PATH = MANIFESTS_DIR / "caption_variants_selected.csv"


def get_labels(dataset: OxfordIIITPet) -> np.ndarray:
    """Extract the integer breed label for every sample in an OxfordIIITPet dataset.

    Uses the dataset's internal `_labels` cache when available (fast path) and
    falls back to iterating the dataset otherwise.

    Args:
        dataset: an OxfordIIITPet dataset instance.

    Returns:
        np.ndarray: integer class label for each sample, in dataset order.
    """
    labels = getattr(dataset, "_labels", None)
    if labels is not None:
        return np.asarray(labels)
    return np.asarray([label for _, label in dataset])


def load_generation_artifacts(
    data_root: Path, split_path: Path, selected_variants_path: Path
) -> tuple[OxfordIIITPet, list[str], np.ndarray, np.ndarray, pd.DataFrame]:
    """Load the dataset, train split and selected captions produced by 02/03.

    Assumes 02_Captioning.ipynb and 03_Rephrasing.ipynb have already been run: the Oxford-IIIT
    Pet trainval split is already cached under `data_root`, 02's Section 1 train/val split is
    already saved at `split_path`, and 03's Section 3.3 best-variant selection is already saved
    at `selected_variants_path` — so everything here is only loaded, never recomputed.

    Args:
        data_root: directory where the Oxford-IIIT Pet dataset is cached.
        split_path: JSON file saved by 02's Section 1, with `train_idx`/`val_idx`.
        selected_variants_path: CSV manifest saved by 03's Section 3.3, one row per image with
            a `selected_variant` column (the best in-band, CLIPScore-passing caption).

    Returns:
        tuple[OxfordIIITPet, list[str], np.ndarray, np.ndarray, pd.DataFrame]: (trainval_dataset,
            class_names, trainval_labels, train_idx, selected_variants).
    """
    trainval_dataset = OxfordIIITPet(root=str(data_root), split="trainval", target_types="category", download=False)
    class_names = trainval_dataset.classes
    trainval_labels = get_labels(trainval_dataset)

    split = json.loads(split_path.read_text())
    train_idx = np.asarray(split["train_idx"])

    selected_variants = pd.read_csv(selected_variants_path)

    return trainval_dataset, class_names, trainval_labels, train_idx, selected_variants

In [6]:
trainval_dataset, class_names, trainval_labels, train_idx, selected_variants = load_generation_artifacts(
    DATA_ROOT, TRAIN_VAL_SPLIT_PATH, SELECTED_VARIANTS_PATH
)
print(
    f"Loaded {len(trainval_dataset)} trainval images ({len(class_names)} classes), "
    f"{len(train_idx)} train indices, and {len(selected_variants)} selected-variant rows "
    f"({selected_variants['selected_variant'].notna().sum()} with a selected variant)"
)

Loaded 3680 trainval images (37 classes), 2944 train indices, and 197 selected-variant rows (176 with a selected variant)


## Section 4 — Image Generation (SD-Turbo img2img)

Generate one synthetic image per `{original_image, selected_variant}` pair from `manifests/caption_variants_selected.csv` (3.3) using `stabilityai/sd-turbo` img2img: the original image is used as the init image and the selected paraphrased caption as the prompt, so the synthetic image stays visually anchored to the original while the caption wording (and therefore the diffusion conditioning) varies. Synthetic images are written to `data/synthetic/<class_label>/<image_id>.jpg`, mirroring the class structure of the original dataset. `IMG_GEN_LIMIT` caps the run for local prototyping, same pattern as `CAPTIONING_LIMIT` in 2.1 — set it to `None` for a full Colab run over every selected variant.

In [ ]:
from diffusers import AutoPipelineForImage2Image

SYNTHETIC_DIR = Path("data/synthetic")
SYNTHETIC_DIR.mkdir(parents=True, exist_ok=True)

SD_TURBO_MODEL_NAME = "stabilityai/sd-turbo"
SD_TURBO_DTYPE = torch.float16 if DEVICE.type == "cuda" else torch.float32
SYNTHETIC_MANIFEST_PATH = MANIFESTS_DIR / "synthetic_images.csv"
AUGMENTED_INDEX_PATH = MANIFESTS_DIR / "augmented_train_index.csv"

IMG2IMG_SIZE = (512, 512)
IMG_GEN_LIMIT = 40  # cap for local prototyping; set to None for a full Colab run
IMG2IMG_STRENGTH = 0.5
IMG2IMG_STEPS = 2


def load_sd_turbo(model_name: str, dtype: torch.dtype, device: torch.device) -> AutoPipelineForImage2Image:
    """Download (if not already cached) and load the SD-Turbo img2img pipeline.

    Args:
        model_name: Hugging Face model id for SD-Turbo.
        dtype: torch dtype for the pipeline weights.
        device: device to move the pipeline to.

    Returns:
        AutoPipelineForImage2Image: the loaded img2img pipeline, with its progress bar disabled.
    """
    pipeline = AutoPipelineForImage2Image.from_pretrained(model_name, torch_dtype=dtype).to(device)
    pipeline.set_progress_bar_config(disable=True)
    return pipeline

In [ ]:
if SYNTHETIC_MANIFEST_PATH.exists():
    print(f"{SYNTHETIC_MANIFEST_PATH} already exists; skipping SD-Turbo download.")
    img2img_pipeline = None
else:
    img2img_pipeline = load_sd_turbo(SD_TURBO_MODEL_NAME, SD_TURBO_DTYPE, DEVICE)

In [ ]:
def generate_synthetic_image(
    image: Image.Image,
    prompt: str,
    pipeline: AutoPipelineForImage2Image,
    seed: int,
    strength: float = 0.5,
    num_inference_steps: int = 2,
    guidance_scale: float = 0.0,
) -> Image.Image:
    """Generate one img2img synthetic image conditioned on an original image and a caption.

    SD-Turbo is a distilled, few-step model: `guidance_scale=0.0` and 1-4 steps are its
    recommended (and only well-tested) operating point, unlike standard Stable Diffusion.

    Args:
        image: the original image, used as the img2img init image.
        prompt: the text prompt steering generation (a Section 3.3 selected caption variant).
        pipeline: the loaded SD-Turbo img2img pipeline.
        seed: seed for this single generation, for reproducibility.
        strength: how much the init image is allowed to change (0 = unchanged, 1 = ignored);
            kept in the 0.5-0.6 range per the implementation plan so results stay recognizable.
        num_inference_steps: number of denoising steps; SD-Turbo needs only 1-4.
        guidance_scale: classifier-free guidance scale; SD-Turbo is trained for 0.0.

    Returns:
        Image.Image: the generated synthetic image, resized to `IMG2IMG_SIZE`.
    """
    generator = torch.Generator(device=DEVICE).manual_seed(seed)
    init_image = image.convert("RGB").resize(IMG2IMG_SIZE)
    result = pipeline(
        prompt=prompt,
        image=init_image,
        strength=strength,
        num_inference_steps=num_inference_steps,
        guidance_scale=guidance_scale,
        generator=generator,
    )
    return result.images[0]

In [ ]:
def build_synthetic_manifest(
    dataset: OxfordIIITPet,
    selected_variants: pd.DataFrame,
    pipeline: AutoPipelineForImage2Image,
    output_dir: Path,
    seed: int,
    strength: float = 0.5,
    num_inference_steps: int = 2,
    limit: int | None = None,
) -> pd.DataFrame:
    """Generate a synthetic image for every image with a Section 3.3 selected variant.

    Images without a selected variant (`selected_variant` is NaN, i.e. nothing passed the Section
    3.3 band/CLIPScore filter) are skipped rather than generated from a forced-pick caption.
    Synthetic images are saved to `output_dir/<class_label>/<image_id>.jpg`, mirroring the
    original dataset's class organization.

    Args:
        dataset: source OxfordIIITPet dataset that `image_id` indexes into.
        selected_variants: output of Section 3.3 (`selected_variants`), with `image_id`, `class_id`,
            `class_label` and `selected_variant`.
        pipeline: the loaded SD-Turbo img2img pipeline.
        output_dir: root directory for the synthetic dataset's class folders.
        seed: base seed; combined with `image_id` so each generation is reproducible but distinct.
        strength: forwarded to `generate_synthetic_image`.
        num_inference_steps: forwarded to `generate_synthetic_image`.
        limit: optional cap on the number of images generated, for local prototyping.

    Returns:
        pd.DataFrame: image_id, class_id, class_label, prompt, synthetic_path for each generated image.
    """
    eligible = selected_variants.dropna(subset=["selected_variant"]).reset_index(drop=True)
    selected_rows = eligible if limit is None else eligible.iloc[:limit]

    rows = []
    for row in tqdm(selected_rows.itertuples(), total=len(selected_rows), desc="Generating synthetic images"):
        class_dir = output_dir / row.class_label
        class_dir.mkdir(parents=True, exist_ok=True)
        synthetic_path = class_dir / f"{row.image_id}.jpg"

        image, _ = dataset[int(row.image_id)]
        synthetic_image = generate_synthetic_image(
            image,
            row.selected_variant,
            pipeline,
            seed=seed + int(row.image_id),
            strength=strength,
            num_inference_steps=num_inference_steps,
        )
        synthetic_image.save(synthetic_path)

        rows.append(
            {
                "image_id": int(row.image_id),
                "class_id": int(row.class_id),
                "class_label": row.class_label,
                "prompt": row.selected_variant,
                "synthetic_path": str(synthetic_path),
            }
        )
    return pd.DataFrame(rows)

In [ ]:
def get_or_create_synthetic_manifest(
    path: Path,
    dataset: OxfordIIITPet,
    selected_variants: pd.DataFrame,
    pipeline: AutoPipelineForImage2Image,
    output_dir: Path,
    seed: int,
    strength: float = 0.5,
    num_inference_steps: int = 2,
    limit: int | None = None,
) -> pd.DataFrame:
    """Reuse the synthetic-image manifest from disk if one exists, otherwise generate and save it.

    Loading from disk avoids rerunning SD-Turbo generation (slow, especially on CPU/MPS) on every
    notebook rerun. The loaded manifest's `image_id` set is validated against the current
    `selected_variants`/`limit` so a stale manifest is caught rather than silently reused, and
    every referenced image file is checked to still exist on disk.

    Args:
        path: destination/source CSV manifest file.
        dataset: source OxfordIIITPet dataset that `image_id` indexes into.
        selected_variants: output of Section 3.3 (`selected_variants`) to generate images for.
        pipeline: the loaded SD-Turbo img2img pipeline.
        output_dir: root directory for the synthetic dataset's class folders.
        seed: base seed forwarded to `build_synthetic_manifest`.
        strength: forwarded to `build_synthetic_manifest`.
        num_inference_steps: forwarded to `build_synthetic_manifest`.
        limit: optional cap on the number of images generated, for local prototyping.

    Returns:
        pd.DataFrame: see `build_synthetic_manifest`.
    """
    eligible = selected_variants.dropna(subset=["selected_variant"])
    expected_ids = set(eligible["image_id"].astype(int)) if limit is None else set(
        eligible["image_id"].astype(int).iloc[:limit]
    )
    if path.exists():
        manifest = pd.read_csv(path)
        paths_exist = manifest["synthetic_path"].map(lambda p: Path(p).exists()).all()
        if set(manifest["image_id"]) == expected_ids and paths_exist:
            print(f"Loaded existing synthetic manifest from {path} ({len(manifest)} images)")
            return manifest
        raise ValueError(f"{path} does not match the current selection/limit, or files are missing; delete it to regenerate.")

    manifest = build_synthetic_manifest(
        dataset, selected_variants, pipeline, output_dir, seed,
        strength=strength, num_inference_steps=num_inference_steps, limit=limit,
    )
    manifest.to_csv(path, index=False)
    print(f"Generated {len(manifest)} synthetic images and saved manifest to {path}")
    return manifest

In [ ]:
synthetic_manifest = get_or_create_synthetic_manifest(
    SYNTHETIC_MANIFEST_PATH,
    trainval_dataset,
    selected_variants,
    img2img_pipeline,
    SYNTHETIC_DIR,
    seed=SEED,
    strength=IMG2IMG_STRENGTH,
    num_inference_steps=IMG2IMG_STEPS,
    limit=IMG_GEN_LIMIT,
)
synthetic_manifest.head()

In [ ]:
def spot_check_synthetic_images(
    dataset: OxfordIIITPet, synthetic_manifest: pd.DataFrame, n: int, seed: int, n_cols: int = 3
) -> None:
    """Display original/synthetic image pairs with their prompt, for a visual QA pass.

    Lets you confirm generated images stay recognizable (same breed, not degenerate/artifacted)
    rather than judging the pipeline on image count alone.

    Args:
        dataset: source OxfordIIITPet dataset that `image_id` indexes into.
        synthetic_manifest: output of `build_synthetic_manifest` / `get_or_create_synthetic_manifest`.
        n: number of image pairs to display.
        seed: seed controlling which rows are sampled.
        n_cols: number of (original, synthetic) pairs per row.
    """
    sample = synthetic_manifest.sample(n=min(n, len(synthetic_manifest)), random_state=seed)
    n_rows = math.ceil(len(sample) / n_cols)

    fig, axes = plt.subplots(n_rows, n_cols * 2, figsize=(6 * n_cols * 2, 6 * n_rows))
    axes = np.atleast_2d(axes).reshape(n_rows, n_cols * 2)

    for i, (_, row) in enumerate(sample.iterrows()):
        r, c = divmod(i, n_cols)
        original, _ = dataset[int(row["image_id"])]
        synthetic = Image.open(row["synthetic_path"])

        axes[r, 2 * c].imshow(original)
        axes[r, 2 * c].set_title(f"{row['class_label']} (original)", fontsize=12)
        axes[r, 2 * c].axis("off")

        axes[r, 2 * c + 1].imshow(synthetic)
        axes[r, 2 * c + 1].set_title(f"synthetic\n{row['prompt']}", fontsize=12)
        axes[r, 2 * c + 1].axis("off")

    for ax in axes.flatten()[2 * len(sample):]:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
spot_check_synthetic_images(trainval_dataset, synthetic_manifest, n=4, seed=32, n_cols=2)

In [ ]:
def build_real_index(indices: np.ndarray, labels: np.ndarray, class_names: list[str]) -> pd.DataFrame:
    """Build a `{source, class_id, class_label, path}` index for a set of real-dataset indices.

    `path` stores the string form of each index rather than a file path: real images are
    resolved by indexing back into their source `OxfordIIITPet` dataset (see
    `ManifestImageDataset` in Section 5), not loaded from disk directly. Shared by
    `build_augmented_train_index` below (for the real half of the augmented training set) and by
    Section 5 (for the baseline training set, and the val/test sets).

    Args:
        indices: dataset indices (into whichever dataset `labels` was computed from).
        labels: full per-sample label array for that dataset (`get_labels` output), indexed by
            `indices`.
        class_names: breed name for each class index.

    Returns:
        pd.DataFrame: one "real" row per index, with `source="real"`.
    """
    class_ids = labels[indices]
    return pd.DataFrame(
        {
            "source": "real",
            "class_id": class_ids,
            "class_label": [class_names[label] for label in class_ids],
            "path": indices.astype(str),
        }
    )


In [ ]:
def build_augmented_train_index(
    train_idx: np.ndarray, class_names: list[str], synthetic_manifest: pd.DataFrame
) -> pd.DataFrame:
    """Build the augmented training index: baseline real training images + synthetic images.

    Val/test stay untouched (real images only) per the plan's fair-comparison requirement — this
    index only ever covers the training split. Section 5 reads this to build the "Run B — Augmented"
    dataset, alongside `train_idx` alone for "Run A — Baseline".

    Args:
        train_idx: indices into `trainval_dataset` for the baseline training split (Section 1).
        class_names: breed name for each class index, to label the real-image rows.
        synthetic_manifest: output of Section 4 (`synthetic_manifest`), one row per generated image.

    Returns:
        pd.DataFrame: `source` ("real"/"synthetic"), `class_id`, `class_label`, and `path` (the
            source dataset index for real images, the file path for synthetic images).
    """
    real_rows = build_real_index(train_idx, get_labels(trainval_dataset), class_names)
    synthetic_rows = pd.DataFrame(
        {
            "source": "synthetic",
            "class_id": synthetic_manifest["class_id"],
            "class_label": synthetic_manifest["class_label"],
            "path": synthetic_manifest["synthetic_path"],
        }
    )
    return pd.concat([real_rows, synthetic_rows], ignore_index=True)

In [ ]:
augmented_train_index = build_augmented_train_index(train_idx, class_names, synthetic_manifest)
augmented_train_index.to_csv(AUGMENTED_INDEX_PATH, index=False)

print(
    f"Augmented training index: {len(augmented_train_index)} rows "
    f"({(augmented_train_index['source'] == 'real').sum()} real + "
    f"{(augmented_train_index['source'] == 'synthetic').sum()} synthetic), "
    f"saved to {AUGMENTED_INDEX_PATH}"
)

### 4.2 Synthetic Image Quality Filtering (CLIPScore)

Reference-free check on the *generated images themselves* — catches img2img outputs where `strength` pushed the result far enough from the source image that class-relevant features got overwritten, which neither the caption-quality filtering (2.2) nor the manual visual QA (4.1) can catch, since both look at captions or a small sample rather than every generated image. Reuses the CLIP model already loaded in 2.2: each synthetic image gets two CLIPScores (against its class name, and against the prompt that generated it), and a per-class real-image CLIPScore baseline gives a threshold that accounts for some breeds being more visually distinctive than others. Flagged images are dropped into a **filtered** augmented training index (`manifests/augmented_train_index_filtered.csv`).

In [ ]:
CLIP_MODEL_NAME = "openai/clip-vit-base-patch32"
CLIP_SCORE_THRESHOLD = 0.7
SYNTHETIC_QUALITY_PATH = MANIFESTS_DIR / "synthetic_images_clip_scored.csv"
REAL_CLIP_BASELINE_PATH = MANIFESTS_DIR / "real_images_clip_baseline.csv"
FILTERED_SYNTHETIC_MANIFEST_PATH = MANIFESTS_DIR / "synthetic_images_filtered.csv"
FILTERED_AUGMENTED_INDEX_PATH = MANIFESTS_DIR / "augmented_train_index_filtered.csv"
REPORTS_DIR = Path("reports")
REPORTS_DIR.mkdir(exist_ok=True)
REAL_CLIP_SAMPLES_PER_CLASS = 10
CLIP_STD_THRESHOLD = 1.5

In [ ]:
from transformers import CLIPModel, CLIPProcessor

def load_clip(model_name: str, device: torch.device) -> tuple[CLIPProcessor, CLIPModel]:
    """Download (if not already cached) and load the CLIP processor/model used for CLIPScore.

    Args:
        model_name: Hugging Face model id for CLIP.
        device: device to move the model to.

    Returns:
        tuple[CLIPProcessor, CLIPModel]: the loaded processor and model, in eval mode.
    """
    processor = CLIPProcessor.from_pretrained(model_name)
    model = CLIPModel.from_pretrained(model_name).to(device)
    model.eval()
    return processor, model

def compute_clip_score(
    image: Image.Image,
    caption: str,
    processor: CLIPProcessor,
    model: CLIPModel,
    device: torch.device,
) -> float:
    """Compute a reference-free CLIPScore between an image and a candidate caption.

    Uses the standard CLIPScore formulation (Hessel et al., 2021): `2.5 * max(cosine_similarity, 0)`.
    Unlike BLEU/ROUGE-L in Section 3.2, this needs no reference caption, so it also works on
    Section 2.1's raw BLIP outputs where there is nothing to compare against yet.

    Args:
        image: input image in PIL format.
        caption: candidate caption to score against the image.
        processor: CLIP processor used to prepare model inputs.
        model: CLIP model used to embed the image and caption.
        device: device the model has been moved to.

    Returns:
        float: the CLIPScore; higher means the caption better matches the image.
    """
    inputs = processor(text=[caption], images=image, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        outputs = model(**inputs)
    image_embeds = outputs.image_embeds / outputs.image_embeds.norm(dim=-1, keepdim=True)
    text_embeds = outputs.text_embeds / outputs.text_embeds.norm(dim=-1, keepdim=True)
    cosine_similarity = (image_embeds * text_embeds).sum(dim=-1).item()
    return 2.5 * max(cosine_similarity, 0.0)

In [ ]:
if SYNTHETIC_QUALITY_PATH.exists() and REAL_CLIP_BASELINE_PATH.exists():
    print(f"{SYNTHETIC_QUALITY_PATH} and {REAL_CLIP_BASELINE_PATH} already exist; skipping CLIP download.")
    clip_processor, clip_model = None, None
else:
    clip_processor, clip_model = load_clip(CLIP_MODEL_NAME, DEVICE)

In [ ]:
def score_synthetic_image_quality(
    synthetic_manifest: pd.DataFrame,
    clip_processor: CLIPProcessor,
    clip_model: CLIPModel,
    device: torch.device,
) -> pd.DataFrame:
    """Score every synthetic image for class-alignment and prompt-alignment via CLIPScore.

    Two independent CLIPScores are computed per image, reusing `compute_clip_score` from 2.2
    (this time scoring a generated image instead of a real one): one against the class name
    ("a photo of a {class_label}"), which checks the generated image still looks like the
    intended breed, and one against the prompt that drove generation, which checks the image
    matches what was actually asked for.

    Args:
        synthetic_manifest: output of Section 4 (`synthetic_manifest`), with `image_id`,
            `class_id`, `class_label`, `prompt`, `synthetic_path`.
        clip_processor: CLIP processor used to prepare model inputs.
        clip_model: CLIP model used to compute CLIPScore.
        device: device the model has been moved to.

    Returns:
        pd.DataFrame: `synthetic_manifest` with added `variant_idx` (always 0, since 3.3 selects
            a single top-1 variant per image), `clip_score_vs_class` and `clip_score_vs_caption`
            columns.
    """
    scored = synthetic_manifest.copy()
    scored["variant_idx"] = 0
    clip_score_vs_class, clip_score_vs_caption = [], []
    for row in tqdm(scored.itertuples(), total=len(scored), desc="Scoring synthetic image quality"):
        synthetic_image = Image.open(row.synthetic_path).convert("RGB")
        class_prompt = f"a photo of a {row.class_label}"
        clip_score_vs_class.append(
            compute_clip_score(synthetic_image, class_prompt, clip_processor, clip_model, device)
        )
        clip_score_vs_caption.append(
            compute_clip_score(synthetic_image, row.prompt, clip_processor, clip_model, device)
        )
    scored["clip_score_vs_class"] = clip_score_vs_class
    scored["clip_score_vs_caption"] = clip_score_vs_caption
    return scored

In [ ]:
def get_or_score_synthetic_quality(
    path: Path,
    synthetic_manifest: pd.DataFrame,
    clip_processor: CLIPProcessor,
    clip_model: CLIPModel,
    device: torch.device,
) -> pd.DataFrame:
    """Reuse persisted synthetic-image CLIPScore quality scores from disk if present, otherwise compute and save them.

    Args:
        path: destination/source CSV file for the scored manifest.
        synthetic_manifest: output of Section 4 (`synthetic_manifest`) to score.
        clip_processor: CLIP processor used to prepare model inputs.
        clip_model: CLIP model used to compute CLIPScore.
        device: device the model has been moved to.

    Returns:
        pd.DataFrame: see `score_synthetic_image_quality`.
    """
    if path.exists():
        scored = pd.read_csv(path)
        expected_ids = set(synthetic_manifest["image_id"].astype(int))
        required_cols = {"variant_idx", "clip_score_vs_class", "clip_score_vs_caption"}
        if required_cols.issubset(scored.columns) and set(scored["image_id"]) == expected_ids:
            print(f"Loaded existing synthetic image quality scores from {path} ({len(scored)} rows)")
            return scored
        raise ValueError(f"{path} does not match the current synthetic manifest; delete it to regenerate.")

    scored = score_synthetic_image_quality(synthetic_manifest, clip_processor, clip_model, device)
    scored.to_csv(path, index=False)
    print(f"Scored synthetic image quality for {len(scored)} images and saved to {path}")
    return scored

In [ ]:
synthetic_quality = get_or_score_synthetic_quality(
    SYNTHETIC_QUALITY_PATH, synthetic_manifest, clip_processor, clip_model, DEVICE
)
synthetic_quality.head()

In [ ]:
def sample_real_class_clip_baseline(
    dataset: OxfordIIITPet,
    indices: np.ndarray,
    labels: np.ndarray,
    class_names: list[str],
    clip_processor: CLIPProcessor,
    clip_model: CLIPModel,
    device: torch.device,
    samples_per_class: int,
    seed: int,
) -> pd.DataFrame:
    """Compute a real-image CLIPScore baseline (image vs. class name) per class.

    Gives a realistic per-class threshold range for `flag_low_quality_synthetic_images` instead
    of picking a single global CLIPScore cutoff blind: "good" alignment between an image and its
    class name varies by breed, since some pets are visually more distinctive than others.

    Args:
        dataset: source OxfordIIITPet dataset that `indices` indexes into.
        indices: candidate real-image indices to sample from (the training split).
        labels: full per-sample label array for `dataset` (`get_labels` output).
        class_names: breed name for each class index.
        clip_processor: CLIP processor used to prepare model inputs.
        clip_model: CLIP model used to compute CLIPScore.
        device: device the model has been moved to.
        samples_per_class: max number of real images sampled per class.
        seed: seed controlling which images are sampled per class.

    Returns:
        pd.DataFrame: `image_id`, `class_id`, `class_label`, `clip_score_vs_class` for the sample.
    """
    rng = np.random.default_rng(seed)
    class_ids = labels[indices]
    rows = []
    for class_id, class_name in enumerate(tqdm(class_names, desc="Sampling real-image CLIP baseline")):
        class_indices = indices[class_ids == class_id]
        sampled = rng.choice(class_indices, size=min(samples_per_class, len(class_indices)), replace=False)
        class_prompt = f"a photo of a {class_name}"
        for image_id in sampled:
            image, _ = dataset[int(image_id)]
            score = compute_clip_score(image, class_prompt, clip_processor, clip_model, device)
            rows.append({"image_id": int(image_id), "class_id": class_id, "class_label": class_name, "clip_score_vs_class": score})
    return pd.DataFrame(rows)

In [ ]:
def get_or_sample_real_clip_baseline(
    path: Path,
    dataset: OxfordIIITPet,
    indices: np.ndarray,
    labels: np.ndarray,
    class_names: list[str],
    clip_processor: CLIPProcessor,
    clip_model: CLIPModel,
    device: torch.device,
    samples_per_class: int,
    seed: int,
) -> pd.DataFrame:
    """Reuse the real-image CLIPScore baseline from disk if present, otherwise sample and save it.

    Args:
        path: destination/source CSV file for the baseline sample.
        dataset: source OxfordIIITPet dataset that `indices` indexes into.
        indices: candidate real-image indices to sample from (the training split).
        labels: full per-sample label array for `dataset` (`get_labels` output).
        class_names: breed name for each class index.
        clip_processor: CLIP processor used to prepare model inputs.
        clip_model: CLIP model used to compute CLIPScore.
        device: device the model has been moved to.
        samples_per_class: max number of real images sampled per class.
        seed: seed controlling which images are sampled per class.

    Returns:
        pd.DataFrame: see `sample_real_class_clip_baseline`.
    """
    if path.exists():
        baseline = pd.read_csv(path)
        print(f"Loaded existing real-image CLIP baseline from {path} ({len(baseline)} rows)")
        return baseline

    baseline = sample_real_class_clip_baseline(
        dataset, indices, labels, class_names, clip_processor, clip_model, device, samples_per_class, seed
    )
    baseline.to_csv(path, index=False)
    print(f"Sampled real-image CLIP baseline ({len(baseline)} rows) and saved to {path}")
    return baseline

In [ ]:
real_clip_baseline = get_or_sample_real_clip_baseline(
    REAL_CLIP_BASELINE_PATH, trainval_dataset, train_idx, trainval_labels, class_names,
    clip_processor, clip_model, DEVICE, REAL_CLIP_SAMPLES_PER_CLASS, SEED,
)
real_clip_baseline.head()

In [ ]:
def flag_low_quality_synthetic_images(
    synthetic_quality: pd.DataFrame, real_clip_baseline: pd.DataFrame, std_threshold: float = 1.5
) -> pd.DataFrame:
    """Flag synthetic images whose class-alignment CLIPScore falls notably below the real-image baseline.

    Args:
        synthetic_quality: output of `score_synthetic_image_quality`, with `clip_score_vs_class`
            per image.
        real_clip_baseline: output of `sample_real_class_clip_baseline`, the per-class real-image
            reference.
        std_threshold: number of real-image standard deviations below the real-image mean that
            counts as "notably below" for a given class.

    Returns:
        pd.DataFrame: `synthetic_quality` with added `real_mean`, `real_std`, `flagged` columns.
    """
    class_stats = real_clip_baseline.groupby("class_id")["clip_score_vs_class"].agg(["mean", "std"]).rename(
        columns={"mean": "real_mean", "std": "real_std"}
    )
    scored = synthetic_quality.merge(class_stats, on="class_id", how="left")
    scored["flagged"] = scored["clip_score_vs_class"] < (scored["real_mean"] - std_threshold * scored["real_std"])
    return scored

In [ ]:
flagged_synthetic = flag_low_quality_synthetic_images(synthetic_quality, real_clip_baseline, CLIP_STD_THRESHOLD)
print(
    f"Flagged {flagged_synthetic['flagged'].sum()} / {len(flagged_synthetic)} synthetic images "
    f"({flagged_synthetic['flagged'].mean():.1%}) as notably below their class's real-image CLIPScore baseline"
)

In [ ]:
def build_real_vs_synthetic_report(flagged_synthetic: pd.DataFrame, real_clip_baseline: pd.DataFrame) -> pd.DataFrame:
    """Build a per-class real-vs-synthetic CLIPScore comparison and flagged-count summary.

    Args:
        flagged_synthetic: output of `flag_low_quality_synthetic_images`.
        real_clip_baseline: output of `sample_real_class_clip_baseline`.

    Returns:
        pd.DataFrame: one row per class with real/synthetic mean+std CLIPScore, flagged count, and
            the real-vs-synthetic gap, sorted by gap descending (largest gap first).
    """
    real_stats = real_clip_baseline.groupby("class_label")["clip_score_vs_class"].agg(["mean", "std"]).rename(
        columns={"mean": "real_mean", "std": "real_std"}
    )
    synth_stats = flagged_synthetic.groupby("class_label").agg(
        synthetic_mean=("clip_score_vs_class", "mean"),
        synthetic_std=("clip_score_vs_class", "std"),
        n_synthetic=("clip_score_vs_class", "size"),
        n_flagged=("flagged", "sum"),
    )
    report = real_stats.join(synth_stats, how="outer")
    report["gap"] = report["real_mean"] - report["synthetic_mean"]
    return report.sort_values("gap", ascending=False)

In [ ]:
real_vs_synthetic_report = build_real_vs_synthetic_report(flagged_synthetic, real_clip_baseline)
real_vs_synthetic_report.to_csv(REPORTS_DIR / "real_vs_synthetic_clip_report.csv")
real_vs_synthetic_report.head(10)

In [ ]:
def build_filtered_synthetic_manifest(flagged_synthetic: pd.DataFrame) -> pd.DataFrame:
    """Drop CLIPScore-flagged rows, keeping only the columns Section 4 produced.

    Args:
        flagged_synthetic: output of `flag_low_quality_synthetic_images`.

    Returns:
        pd.DataFrame: `synthetic_manifest`-shaped subset (`image_id`, `class_id`, `class_label`,
            `prompt`, `synthetic_path`) with flagged rows removed.
    """
    kept = flagged_synthetic[~flagged_synthetic["flagged"]].reset_index(drop=True)
    return kept[["image_id", "class_id", "class_label", "prompt", "synthetic_path"]]

In [ ]:
filtered_synthetic_manifest = build_filtered_synthetic_manifest(flagged_synthetic)
filtered_synthetic_manifest.to_csv(FILTERED_SYNTHETIC_MANIFEST_PATH, index=False)

n_dropped = len(synthetic_manifest) - len(filtered_synthetic_manifest)
print(
    f"Synthetic images after CLIPScore filtering: {len(filtered_synthetic_manifest)} / {len(synthetic_manifest)} "
    f"({n_dropped} dropped, {n_dropped / len(synthetic_manifest):.1%}); saved to {FILTERED_SYNTHETIC_MANIFEST_PATH}"
)

In [ ]:
augmented_train_index_filtered = build_augmented_train_index(train_idx, class_names, filtered_synthetic_manifest)
augmented_train_index_filtered.to_csv(FILTERED_AUGMENTED_INDEX_PATH, index=False)

print(
    f"Filtered augmented training index: {len(augmented_train_index_filtered)} rows "
    f"({(augmented_train_index_filtered['source'] == 'real').sum()} real + "
    f"{(augmented_train_index_filtered['source'] == 'synthetic').sum()} synthetic), "
    f"saved to {FILTERED_AUGMENTED_INDEX_PATH}"
)

In [ ]:
print("Section 4 complete — artifacts on disk:")
print(f"  {SYNTHETIC_DIR}/ : {len(synthetic_manifest)} generated images, one folder per breed")
print(f"  {SYNTHETIC_MANIFEST_PATH} : {len(synthetic_manifest)} synthetic images and their prompts")
print(f"  {AUGMENTED_INDEX_PATH} : {len(augmented_train_index)} training rows (all synthetic kept)")
print(f"  {SYNTHETIC_QUALITY_PATH} : CLIPScores for every synthetic image")
print(f"  {REAL_CLIP_BASELINE_PATH} : per-class real-image CLIPScore baseline")
print(f"  {FILTERED_SYNTHETIC_MANIFEST_PATH} : {len(filtered_synthetic_manifest)} synthetic images surviving the filter")
print(f"  {FILTERED_AUGMENTED_INDEX_PATH} : {len(augmented_train_index_filtered)} training rows (filtered)")
print(f"  {REPORTS_DIR}/real_vs_synthetic_clip_report.csv : per-class real-vs-synthetic gap")
print("\nNext: run 05_ClassifierTraining.ipynb.")